# Blood signal through the adult temple, without contrast agent

This notebook measures **how strong the echo from moving blood is, compared with the electronic noise, on a single ultrasound receive element**, when the probe sits on the temple of an adult and no contrast agent (microbubbles) is in the blood.

The data are public recordings from the Open-H-RF dataset (Resolve Stroke contribution, CC BY 4.0): adult ICU/CCU patients from the SCULPT study (the dataset card says "ICU/CCU patients"; the study registration, NCT07324421, requires age ≥ 18), imaged through the temporal bone with a 2 MHz matrix probe, 4000 frames per second. Each recording contains a 1-second clip taken **before** a microbubble injection reached the brain. That clip is what we analyse. A later clip (with bubbles) is used only as a check that the "before" clip really has no bubbles in it.

Two scripts do all the work; this notebook just calls them and explains each step:
* `get_data.py` downloads only the pieces of each file that we need (about 1 GB per recording instead of 5.3 GB) and checks them.
* `temple_snr.py` computes the numbers and draws one figure.

## Settings

`N_RECORDINGS = 4` is the quick run (about 1 GB each). 10 gives the first recording of every subject. **20 is the full study** (10 subjects x 2 recordings, about 19 GB): each subject's value is then the linear mean of its 2 recordings. The full study gave -22.4 dB at 37.5-42.5 mm (90 % CI -24.8 to -18.9 dB), with a formal verdict of FAIL(WIDE) because the 40-45 mm CI was 6.45 dB wide (limit 6.0); see README.md. Data and results go into folders next to this notebook unless you change the paths.

In [ ]:
import os, tempfile, json
N_RECORDINGS = 4
DATA_DIR = "data"
OUT_DIR = "results"

## Step 1: download and check the data

For each recording, the script reads the file's internal index over the web, works out exactly which compressed byte ranges hold the 4000 "before" frames (plus 256 frames of the contrast clip), and downloads only those. Then it checks that:
* every frame decompresses to the right shape;
* what is stored locally is byte-for-byte what was downloaded;
* 32 randomly chosen frames, downloaded a second time, are byte-identical;
* the file on the server still has the size the index was read from.

If the download stops, run the cell again: finished pieces are kept and skipped.

In [ ]:
import get_data
get_data.main(["--n", str(N_RECORDINGS), "--data-dir", DATA_DIR])

## Step 2 (optional, about 20-50 s): check the method on fake data with a known answer

We build a synthetic recording with white noise, a "blood" signal at exactly -22 dB, the same artefact lines found in the real data, and a strong static echo. The method should give back -22 dB (within 0.5 dB) both with the main estimator (E) and with the independent cross-check (X), on the 30-60 mm gate. The 5-mm gates (P1, P2) are printed as "(info, not tested)": on one synthetic recording they scatter by about ±0.5-1 dB from sampling noise alone.

In [ ]:
import temple_snr as T
with tempfile.TemporaryDirectory() as td:
    T.selftest(td)

## Step 3: the analysis settings

All choices are in one dictionary, `CONFIG`, at the top of `temple_snr.py`. In plain words:
* **Depth gates.** The headline gate P1 is 37.5 to 42.5 mm below the skin; P2 is 40 to 45 mm. For each receive element we pick the time samples whose echoes come from that depth.
* **Slow-time spectrum.** For each time sample and element we look at how the signal changes from frame to frame (4000 frames). Moving blood shows up as Doppler frequencies; still tissue and bone sit at 0 Hz; electronic noise is spread evenly over all frequencies.
* **Clutter filter.** Everything below 100 Hz (bone and tissue echo, which is thousands of times stronger than blood) is ignored.
* **Artefact lines.** The scanner leaves narrow lines at exactly 1000 and 2000 Hz (a quarter and half of the frame rate). They are not blood, so they are cut out.
* **Noise.** Between 1500 and 2000 Hz there is essentially no blood at these depths, so the level there is the noise floor.
* **SNR.** Blood power = power between 100 and 1500 Hz that sits above the noise floor. SNR = blood power / noise power, per element, per transmitted pulse.
* **Contamination check.** In the clip with contrast, the same quantity must be at least 6 dB (4x) higher than in the "before" clip; otherwise the recording is not used for the summary.

In [ ]:
T.CONFIG

## Step 4: run it

Each recording takes a few seconds. The table gives, per recording, the contamination check (`Cmin`), whether the recording is included, the number of valid elements, and the SNR in dB in each gate, for the main estimator (E), the cross-check (X), and E without the artefact-line notch (to show how much the lines matter). A second table gives one value per subject (the linear mean of its included recordings). The last lines give the median over subjects with a 90 % bootstrap interval, the verdict per gate, and the overall verdict (both P1 and P2 must pass: at least 8 of 10 subjects, CI narrower than 6 dB, E and X within 3 dB). A quick run with 4 recordings always says FEW.

In [ ]:
results, pops = T.run(DATA_DIR, OUT_DIR)

## Step 5: the figure

Left: SNR against depth for each recording (dotted grey line with x markers = excluded by the contamination check), and the median over included subjects (black). Right: the average slow-time spectrum in the headline gate, divided by the noise floor, shown **without** the artefact-line notch. Blood is the small hump above 0 dB inside the red band, mostly just above the grey clutter band; the noise band is blue. The narrow spikes at ±1000 Hz and -2000 Hz (PRF/2, at the left edge after re-ordering the frequencies) are the scanner's artefact lines. No lines are drawn for them; they are cut out (±8 Hz) before the SNR is computed.

In [ ]:
from IPython.display import Image
Image(os.path.join(OUT_DIR, "temple_snr.png"))

## How to read the number

* It is the SNR of **one receive element of this probe** (one of the 256 receive channels of a 2 MHz, 32 x 32, 0.5 mm-pitch matrix probe, which the dataset card calls "virtual elements" (receive sub-apertures flattened); their positions are 256 distinct points on the 0.5 mm grid, so we treat each as one element, with one unfocused diverging wave per frame), for **contrast-free blood**, **per transmitted pulse**, at 40 mm through the temple, before any beamforming or averaging over frames.
* A value around -20 dB means the blood echo on one element for one pulse is about 100 times weaker than the noise. Imaging still works because thousands of pulses and hundreds of elements are combined.
* It is **not** the SNR of a beamformed image, not the SNR of another probe, frequency or transmit power, and not a measure of how well any particular device would work. Converting it to another probe needs extra assumptions that are not tested here.
* Anything that moves faster than the clutter filter but is not blood (tissue pulsation, patient motion) is counted as blood, so the number is an upper bound in that respect; very fast blood (above about 0.57 m/s) is missed.
* Slow blood is missed too: the 100 Hz clutter filter removes blood moving slower than about 0.04 m/s along the beam, which includes most microvascular and perfusion flow. In the full study, moving the filter to 200 Hz lowered the result by 1.2 dB, and dropping the artefact-line notch raised it by 2.0 dB.

In [ ]:
print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk in ("n_subjects", "median_db", "ci90_db", "ci_width_db", "verdict")}
                  for k, v in json.load(open(os.path.join(OUT_DIR, "temple_snr_results.json")))["population"].items()}, indent=1))